 - Spark configured with local[n] can simulate shuffles 

In [ ]:
from contexttimer import Timer
import kagglehub, pandas as pd
from pathlib import Path
import humanize
import time
import importlib

from squid.jupyter.extensions import viewdf, viewdf_pandas


from squid.spark.session import get_spark

from pyspark.sql import functions as F
import pandas as pd
from rich.console import Console

from sparkmonitor import kernelextension

In [ ]:
monitor = kernelextension.monitor
if not hasattr(monitor, "comm"):
    print(
        "SparkMonitor frontend is not connected.\n"
        "Reload the JupyterLab page with Ctrl+Shift+R."
    )
    raise SystemExit

In [ ]:
_console = Console(force_jupyter=False)
print = _console.print

In [2]:
def show_spark_config(names: list[str]):
    for name in names:
        value = spark.conf.get(name)
        print(f"{name}: {value}")

In [ ]:
import argparse

parser = argparse.ArgumentParser()

parser.add_argument(
    "--data-format",
    choices=["iceberg", "delta", "none"],
    default="none",
)

args, unknown_args = parser.parse_known_args()

print(f"Args: {args} - unknown_args: {unknown_args}")

data_format = (
    None
    if args.data_format == "none"
    else args.data_format
)


In [ ]:
sc, spark = get_spark(
    data_format=data_format
)

In [4]:
if False:
    show_spark_config([
        "spark.sql.extensions",
        "spark.sql.catalog.spark_catalog",
        "spark.sql.catalog.iceberg",
        "spark.sql.catalog.iceberg.warehouse",
        "spark.sql.catalog.iceberg.type",
        "spark.sql.warehouse.dir",
        
        "spark.sql.shuffle.partitions",
    ])
    
    # For local mode, the number inside local[n] represents local worker threads
    print(f"spark.sparkContext.master: {spark.sparkContext.master}")

In [ ]:
%%sql
SHOW CATALOGS;
SHOW DATABASES;
-- SHOW NAMESPACES IN spark_catalog

In [ ]:
catalog_name = "local" if data_format == "iceberg" else "spark_catalog"
catalog_manager = spark._jsparkSession.sessionState().catalogManager()
catalog = catalog_manager.catalog(catalog_name)
current_catalog = catalog_manager.currentCatalog()


print(
    f"Catalog {catalog_name}: {catalog.getClass().getName()}"
    f". Current: {current_catalog.getClass().getName()}"
)

In [ ]:
display(spark)

In [ ]:
!free -h

In [ ]:
##%%sql
#CREATE OR REPLACE FUNCTION humanize_number(n BIGINT)
#RETURNS STRING
#RETURN
#    CASE
#        WHEN n >= 1000000000 THEN CONCAT(ROUND(n / 1000000000.0, 1), ' billion')
#        WHEN n >= 1000000    THEN CONCAT(ROUND(n / 1000000.0, 1), ' million')
#        WHEN n >= 1000       THEN CONCAT(ROUND(n / 1000.0, 1), ' thousand')
#       ELSE CAST(n AS STRING)
#    END;